# 🧠 EX54: การฝึกสอนต่อจากจุดที่ถูกขัดจังหวะ (Resuming Interrupted Training)

การเทรนอาจถูกขัดจังหวะด้วย OOM, ไฟดับ หรือ cloud preemption
`last.pt` บันทึก **สถานะการเทรนทั้งหมด** ไว้ครบถ้วน

## สิ่งที่อยู่ภายใน `last.pt`
```python
{
  'epoch':        42,     # รอบล่าสุดที่เสร็จ (นับจาก 0)
  'best_fitness': 0.712,  # คะแนน fitness ดีที่สุด
  'model':        ...,    # weight state_dict
  'ema':          ...,    # EMA state_dict (→ best.pt)
  'optimizer':    ...,    # tensor m_t, v_t ของ AdamW
  'train_args':   {...},  # config เดิม
}
```

### ทำไม Optimizer State สำคัญ?
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
การสูญเสีย $m_t, v_t$ รีเซ็ต adaptive LR → ประสิทธิภาพตกชั่วคราว

- ✅ ใช้ `last.pt` เสมอ
- ❌ อย่าใช้ `best.pt` (ขาด optimizer state)

## 🔗 ลิงก์
- [[EX53_Training_Settings_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os
import torch, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from solution import resume_yolo_training
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")
print("\n--- เริ่มการตรวจสอบ ---")

# ขั้นที่ 1: สร้าง checkpoint ด้วยการเทรน 2 epoch
print("[ขั้นที่ 1] เทรน 2 epoch เพื่อสร้าง last.pt...")
base = YOLO("yolo11n.pt")
base_res = base.train(data="coco8.yaml", epochs=2, imgsz=320, batch=8, device=device, verbose=False)
ckpt = str(base_res.save_dir / "weights" / "last.pt")
print(f"Checkpoint: {ckpt}")

# ขั้นที่ 2: ตรวจสอบ checkpoint
print("\n[ขั้นที่ 2] ตรวจสอบ checkpoint...")
ck = torch.load(ckpt, map_location="cpu")
print(f"  Keys: {list(ck.keys())}")
print(f"  Epoch เสร็จ: {ck.get('epoch','N/A')}")
print(f"  Best fitness: {ck.get('best_fitness',0.0):.4f}")
print(f"  Optimizer state: {'มี ✅' if ck.get('optimizer') is not None else 'ขาดหาย ❌'}")
ta = ck.get("train_args", {})
print(f"  Config: epochs={ta.get('epochs','?')}, batch={ta.get('batch','?')}")

# ขั้นที่ 3: Resume
print("\n[ขั้นที่ 3] กำลัง resume...")
res = resume_yolo_training(ckpt)
print(f"บันทึกผลที่: {res.save_dir}")
print("--- สิ้นสุดการตรวจสอบ ---")

csv = res.save_dir / "results.csv"
if csv.exists():
    df = pd.read_csv(csv); df.columns=[c.strip() for c in df.columns]
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    plt.figure(figsize=(8,4))
    for col in loss_cols: plt.plot(df["epoch"], df[col], label=col, linewidth=2)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss หลัง Resume")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

del base
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
